# PRIMEROS PASOS EN PYTHON.
En este script crearemos un detector de las clases que consideremos en tiempo real.
Para ello usaremos el modelo YOLO("You Only Look Once"), este modelo está pre-entrenado con un set de imágenes y es capaz de detectar hasta 80 clases distintas.
Realizaremos detecciones en tiempo real y lo mostraremos en una ventana emergente señalando la clase y el lugar donde se encuentra el objeto de interés. Dicha predicción se podrá realizar desde la webcam de nuestro propio portátil o desde una cam pública.

##### Importación de librerías

In [ ]:
import cv2                              # OpenCV. Librería especializada en tratamiento de Imágenes.
import os                               # Sistema Operativo. Librería para llamadas al sistema.
from ultralytics import YOLO            # Ultralytics. Librería donde se incluye el modelo que vamos a usar.
from dotenv import load_dotenv          # .ENV. Librería para la carga y manipulación del Entorno.

##### Carga de entorno

In [ ]:
load_dotenv()                           # Carga de las variables de Entorno.

##### Clases a detectar.
Define que clases vamos a detectar en esta actividad. Puedes encontrar todas en la siguiente web: [Clases YOLO](https://docs.ultralytics.com/es/datasets/detect)

In [ ]:
# TODO: Declara una lista con las clases que deseas detectar.
target_classes = [ "person" ] 

##### Umbral de detección.
Como hemos visto, el modelo nos devuelve las clases detectadas junto a una confianza de detección, este umbral nos ayudará a eliminar falsos positivos y en un futuro eliminaremos avisos que no se correspondan con la realidad.

In [ ]:
# TODO: Establece un umbral mínimo (0-1) para eliminar detecciones con mejor confianza a este.
UMBRAL = 0.5

##### Carga del modelo.
Dadas las características del PC en el que estamos trabajando debemos buscar que modelo se ajusta mejor y da un rendimiento aceptable. 
Para ello nos guiaremos de la siguiente tabla:



| Tamaño del Modelo | Código | Velocidad (Inferencia) | Precisión (mAP) | VRAM mínima recomendada |
| :--- | :--- | :--- | :--- | :--- |
| **Nano** | `n` | ⚡ Muy Alta | 📉 Baja/Aceptable | No requiere (Ideal CPU) |
| **Small** | `s` | 🚀 Alta | 📊 Moderada | ~2 - 4 GB VRAM |
| **Medium** | `m` | ⏱️ Media | 📈 Alta | ~4 - 6 GB VRAM |
| **Large** | `l` | 🐢 Baja | 🎯 Muy Alta | ~8 - 12 GB VRAM |
| **XLarge** | `x` | 🐌 Muy Baja | 🏆 Máxima | > 12 GB VRAM |





In [ ]:
print("Cargando modelo YOLO...")
# TODO: Carga el modelo que mejor se ajuste a tu PC.
model = YOLO('yolov8n.pt')

In [ ]:
cam_env = os.getenv("CAMARA_FUENTE", "0")
camara_fuente = int(cam_env) if cam_env.isdigit() else cam_env

print(f"Camara elegida: {camara_fuente}")

##### Bucle principal.
En este bucle se realiza la detección en sí. Primero deberemos de obtener la imagen en la que aplicaremos la inferencia, después le preguntaremos al modelo si detecta algo, una vez obtenido todas las clases presentes en la imagen, filtraremos por las clases de interés y mostraremos por pantalla aquellas cuya confianza supere el umbral.

In [ ]:
cap = cv2.VideoCapture(camara_fuente)
print(f"Iniciando cámara ({camara_fuente})... (Pulsa 'q' para salir)")

while True:
    ret, frame = cap.read()
    if not ret:
        print("Esperando señal de la cámara...")
        continue

    # TODO: Pasa al modelo la instantánea que debe de procesar. Recuerda que los resultados que 
    #       nos interesan están en la primera posición del array.
    results = model(frame)[0]

    # TODO: Itera sobre las cajas (.boxes) detectadas en tus resultados
    
    for result in results.boxes: 
        
        # Extracción de las características que nos importan.
        cls_id = int(result.cls)
        conf = float(result.conf)
        label = model.names[cls_id]

        # TODO: ¿Es lo que buscamos? 
        # Comprueba si la etiqueta (label) está en tus target_classes 
        # Y si la confianza (conf) es mayor o igual a tu UMBRAL
        if label in target_classes and conf >= UMBRAL: 
            
            # Extraer coordenadas de la caja
            x1, y1, x2, y2 = map(int, result.xyxy[0])
            
            # Dibujamos en las coordenadas donde encontramos el objeto de interés un rectángulo junto la clase a la que
            # pertenece y la confianza de la prediccion.                                                                  
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 4)
            
            texto = f"{label} {conf:.2f}"
            (w, h), base = cv2.getTextSize(texto, cv2.FONT_HERSHEY_SIMPLEX, 1.0, 2)
            cv2.rectangle(frame, (x1, y1), (x1 + w, y1 - h - base), (0, 255, 0), cv2.FILLED)
            cv2.putText(frame, texto, (x1, y1 - base), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 2)

    cv2.imshow("Fase 1: Seguimiento IA", frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break



##### Destruir los objetos que se han quedado bloquedos.
Al abortar el código anterior se nos queda una ventana inaccesible y la webcam encendida. Para desbloquear ambos, debemos de destruir las instancias que antes hemos usado.

In [11]:
cap.release()
cv2.destroyAllWindows()